# WESAD Stress Detection — Improved SSL with BYOL + Temporal Neighbours

**Improvements over SimCLR baseline:**

| Aspect | SimCLR baseline | This notebook |
|--------|-----------------|---------------|
| Algorithm | NT-Xent (needs large negatives) | **BYOL** (no negatives, EMA target) |
| Positive pairs | Two augmented copies of same window | **Temporally adjacent windows** same subject |
| Augmentation | 4 functions, 1 applied per view | **6 functions, 2 composed per view** |
| EMA schedule | Fixed τ | **Cosine τ schedule 0.990 → 0.999** |
| Pretraining data | Train subjects only | **All 15 subjects** (unlabelled) |
| Encoder | 3-layer CNN, 128-dim | **4-layer residual CNN, 256-dim** |
| Ablations | None | **Temporal-K + Augmentation strategy** |

## Before you run

1. Mount Google Drive and set `DATA_ROOT` (cell 4).
2. `scipy` is used for Gaussian smoothing — already in pip check.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted.")
except Exception:
    print("Not in Colab — skipping mount.")

In [ ]:
import sys, subprocess

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, name in [("numpy","numpy"),("scipy","scipy"),("pandas","pandas"),
                   ("scikit-learn","sklearn"),("matplotlib","matplotlib"),
                   ("seaborn","seaborn"),("tqdm","tqdm")]:
    try: __import__(name)
    except Exception: pip_install(pkg)

print("Dependencies ready.")

In [ ]:
import os, glob, pickle, random, math, copy
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.ndimage import gaussian_filter1d

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score,
                              confusion_matrix, classification_report)

# ── Change this ───────────────────────────────────────────────────────────────
DATA_ROOT  = "/content/drive/MyDrive/PhD/Courses/AdvML/WESAD"
OUTPUT_DIR = "/content/wesad_byol_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42

ORIG_FS    = 700
DOWNSAMPLE = 10
FS         = ORIG_FS // DOWNSAMPLE   # 70 Hz

WINDOW_SEC = 30
STEP_SEC   = 15
WIN        = WINDOW_SEC * FS
STEP       = STEP_SEC   * FS
PURITY     = 0.90

LABEL_MAP   = {1: 0, 2: 1, 3: 2}
ID2LABEL    = {0: "baseline", 1: "stress", 2: "amusement"}
NUM_CLASSES = 3

# ── Hyper-parameters ─────────────────────────────────────────────────────────
EMB_DIM    = 256
PROJ_DIM   = 256
PRED_DIM   = 128
SSL_BATCH  = 64           # BYOL works fine with small batches
SUP_BATCH  = 128
SSL_EPOCHS = 100
LINEAR_EPOCHS = 30
SUP_EPOCHS    = 30
LR_SSL     = 3e-4
LR_LINEAR  = 1e-3
LR_SUP     = 1e-3
WEIGHT_DECAY  = 1e-4
WARMUP_EPOCHS = 10
MIN_LR_FACTOR = 1e-2
EMA_START  = 0.990        # tau at epoch 0
EMA_END    = 0.999        # tau at final epoch (cosine schedule)
TEMPORAL_K = 8            # default positive-pair temporal distance

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)

## Data loading (identical pipeline to SimCLR baseline)

In [ ]:
def get_key(d, keys):
    for k in keys:
        if k in d: return d[k]
    raise KeyError(f"None of {keys} found")


def load_subject_pkl(pkl_path):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f, encoding="latin1")
    chest = data["signal"]["chest"]
    y     = np.asarray(data["label"]).reshape(-1)
    acc   = np.asarray(get_key(chest, ["ACC"])).astype(np.float32)
    ecg   = np.asarray(get_key(chest, ["ECG"])).reshape(-1,1).astype(np.float32)
    eda   = np.asarray(get_key(chest, ["EDA"])).reshape(-1,1).astype(np.float32)
    emg   = np.asarray(get_key(chest, ["EMG"])).reshape(-1,1).astype(np.float32)
    resp  = np.asarray(get_key(chest, ["Resp","RESP"])).reshape(-1,1).astype(np.float32)
    temp  = np.asarray(get_key(chest, ["Temp","TEMP"])).reshape(-1,1).astype(np.float32)
    return np.concatenate([acc, ecg, eda, emg, resp, temp], axis=1), y


def make_windows(x, y, win, step, label_map, purity=0.90):
    Xw, yw = [], []
    for s in range(0, len(y) - win + 1, step):
        e = s + win
        seg = y[s:e]
        vals, cnts = np.unique(seg, return_counts=True)
        maj = vals[np.argmax(cnts)]
        if maj not in label_map or cnts.max()/len(seg) < purity: continue
        Xw.append(x[s:e]); yw.append(label_map[maj])
    if not Xw:
        return np.empty((0,win,x.shape[1]),np.float32), np.empty((0,),np.int64)
    return np.stack(Xw).astype(np.float32), np.array(yw,np.int64)


pkl_files = sorted(glob.glob(os.path.join(DATA_ROOT, "S*", "S*.pkl")))
print(f"Found {len(pkl_files)} subject files.")
if not pkl_files:
    raise FileNotFoundError(f"No pkl files under {DATA_ROOT}/S*/S*.pkl")

In [ ]:
X_all_list, y_all_list, groups_all = [], [], []
for pkl_path in pkl_files:
    sid = os.path.basename(os.path.dirname(pkl_path))
    x, y = load_subject_pkl(pkl_path)
    x, y = x[::DOWNSAMPLE], y[::DOWNSAMPLE]
    Xw, yw = make_windows(x, y, WIN, STEP, LABEL_MAP, PURITY)
    if not len(Xw): print(f"  {sid}: skipped"); continue
    X_all_list.append(Xw); y_all_list.append(yw)
    groups_all.extend([sid]*len(yw))
    print(f"  {sid}: {len(yw)} windows")

X_all      = np.concatenate(X_all_list)
y_all      = np.concatenate(y_all_list)
groups_all = np.array(groups_all)
N_CH       = X_all.shape[2]
print(f"\nAll subjects: {X_all.shape}  classes: {Counter(y_all)}")

## Subject-level split

**SSL pretraining uses ALL 15 subjects (unlabelled)** — valid because BYOL uses no labels.
Labelled evaluation uses only the train-subject split.

In [ ]:
gss = GroupShuffleSplit(1, test_size=0.20, random_state=SEED)
tr_idx, te_idx = next(gss.split(X_all, y_all, groups_all))
X_tr_full, y_tr_full = X_all[tr_idx], y_all[tr_idx]
g_tr_full             = groups_all[tr_idx]
X_test, y_test        = X_all[te_idx], y_all[te_idx]

gss2 = GroupShuffleSplit(1, test_size=0.15, random_state=SEED)
tri, vai = next(gss2.split(X_tr_full, y_tr_full, g_tr_full))
X_train, y_train = X_tr_full[tri], y_tr_full[tri]
X_val,   y_val   = X_tr_full[vai], y_tr_full[vai]
g_val            = g_tr_full[vai]


def standardize(Xtr, Xva, Xte):
    mu  = Xtr.mean((0,1), keepdims=True)
    std = Xtr.std((0,1),  keepdims=True) + 1e-6
    return (Xtr-mu)/std, (Xva-mu)/std, (Xte-mu)/std

X_train, X_val, X_test = standardize(X_train, X_val, X_test)

mu_tr  = X_tr_full[tri].mean((0,1), keepdims=True)
std_tr = X_tr_full[tri].std((0,1),  keepdims=True) + 1e-6
X_all_norm = (X_all - mu_tr) / std_tr    # standardised ALL data for SSL

print(f"Train {X_train.shape}  Val {X_val.shape}  Test {X_test.shape}")
print(f"SSL pretrain pool (all subjects): {X_all_norm.shape}")

## Augmentation Library

Six physiologically-motivated augmentations:

| Function | What it simulates |
|----------|-------------------|
| `jitter` | Sensor noise (Gaussian) |
| `scaling` | Gain variation across channels |
| `time_mask` | Sensor dropout / lost packets |
| `freq_mask` | Band-limited interference (power-line noise, motion artefact) |
| `gaussian_smooth` | Low-pass filter / lower effective sampling rate |
| `channel_dropout` | Complete channel failure (electrode detachment) |

**Composition strategy:** each view applies **2 randomly chosen augmentations in sequence**,
giving $\binom{6}{2}=15$ distinct view pairs — far richer than a single augmentation.

In [ ]:
# ── 6 augmentation functions ─────────────────────────────────────────────────

def jitter(x, sigma=0.03):
    """Additive Gaussian noise."""
    return (x + np.random.normal(0, sigma, x.shape)).astype(np.float32)


def scaling(x, sigma=0.10):
    """Random per-channel amplitude scaling."""
    f = np.random.normal(1.0, sigma, (1, x.shape[1])).astype(np.float32)
    return (x * f).astype(np.float32)


def time_mask(x, max_frac=0.12):
    """Zero a random contiguous temporal segment."""
    y = x.copy(); T = y.shape[0]
    w = max(1, int(T * np.random.uniform(0.03, max_frac)))
    s = np.random.randint(0, max(1, T - w + 1))
    y[s:s+w] = 0.0
    return y


def freq_mask(x, max_frac=0.20):
    """Zero a random frequency band in every channel (FFT domain)."""
    y = x.copy(); T = y.shape[0]
    for ch in range(y.shape[1]):
        Xf = np.fft.rfft(y[:, ch])
        nf = len(Xf)
        w  = max(1, int(nf * np.random.uniform(0.05, max_frac)))
        s  = np.random.randint(0, max(1, nf - w + 1))
        Xf[s:s+w] = 0.0
        y[:, ch]  = np.fft.irfft(Xf, n=T)
    return y.astype(np.float32)


def gaussian_smooth(x, sigma_range=(1.0, 5.0)):
    """Gaussian low-pass filter per channel — simulates lower effective FS."""
    sigma = np.random.uniform(*sigma_range)
    y = np.stack([gaussian_filter1d(x[:, ch], sigma=sigma)
                  for ch in range(x.shape[1])], axis=1)
    return y.astype(np.float32)


def channel_dropout(x, p=0.25):
    """Zero entire channels independently with probability p."""
    y = x.copy()
    mask = (np.random.rand(x.shape[1]) > p).astype(np.float32)
    return (y * mask[None, :]).astype(np.float32)


# ── Composed augmentation (2 draws without replacement) ──────────────────────
AUG_POOL = [jitter, scaling, time_mask, freq_mask, gaussian_smooth, channel_dropout]


def compose_augment(x, n=2):
    """Apply n randomly sampled augmentations in sequence."""
    chosen = random.sample(AUG_POOL, n)
    y = x.copy()
    for f in chosen:
        y = f(y)
    return y


# Quick visual sanity-check
sample = X_train[0]
fig, axes = plt.subplots(2, 3, figsize=(14, 5))
names = ["jitter", "scaling", "time_mask", "freq_mask", "gaussian_smooth", "channel_dropout"]
funcs = [jitter,   scaling,   time_mask,   freq_mask,   gaussian_smooth,   channel_dropout]
for ax, name, fn in zip(axes.flat, names, funcs):
    ax.plot(sample[:, 0], alpha=0.5, label="original")
    ax.plot(fn(sample)[:, 0], alpha=0.8, label=name)
    ax.set_title(name); ax.legend(fontsize=7)
plt.suptitle("Augmented views of ECG channel (Ch 0)")
plt.tight_layout(); plt.show()

## BYOL Model (4-layer residual encoder, 256-dim)

In [ ]:
class ResBlock1D(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm1d(ch), nn.ReLU(),
            nn.Conv1d(ch, ch, 3, padding=1, bias=False), nn.BatchNorm1d(ch))
        self.act = nn.ReLU()
    def forward(self, x): return self.act(x + self.net(x))


class Encoder1D(nn.Module):
    def __init__(self, in_ch=8, emb_dim=256):
        super().__init__()
        self.stem   = nn.Sequential(nn.Conv1d(in_ch,64,7,padding=3,bias=False),
                                    nn.BatchNorm1d(64),nn.ReLU())
        self.layer1 = nn.Sequential(nn.Conv1d(64,128,3,padding=1,bias=False),
                                    nn.BatchNorm1d(128),nn.ReLU(),ResBlock1D(128))
        self.layer2 = nn.Sequential(nn.Conv1d(128,256,3,padding=1,bias=False),
                                    nn.BatchNorm1d(256),nn.ReLU(),ResBlock1D(256))
        self.pool   = nn.AdaptiveAvgPool1d(1)
        self.fc     = nn.Linear(256, emb_dim)
    def forward(self, x):
        x = x.transpose(1,2)
        x = self.stem(x); x = self.layer1(x); x = self.layer2(x)
        return F.normalize(self.fc(self.pool(x).squeeze(-1)), dim=1)


class MLP(nn.Module):
    def __init__(self, in_d, hid_d, out_d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_d,hid_d,bias=False),
                                  nn.BatchNorm1d(hid_d),nn.ReLU(),
                                  nn.Linear(hid_d,out_d))
    def forward(self, x): return F.normalize(self.net(x), dim=1)


class BYOL(nn.Module):
    def __init__(self, in_ch=8, emb_dim=256, proj_dim=256,
                 pred_dim=128, ema_decay=EMA_START):
        super().__init__()
        self.ema_decay   = ema_decay
        self.online_enc  = Encoder1D(in_ch, emb_dim)
        self.online_proj = MLP(emb_dim, proj_dim*2, proj_dim)
        self.predictor   = MLP(proj_dim, pred_dim,  proj_dim)
        self.target_enc  = copy.deepcopy(self.online_enc)
        self.target_proj = copy.deepcopy(self.online_proj)
        for p in self.target_enc.parameters():  p.requires_grad_(False)
        for p in self.target_proj.parameters(): p.requires_grad_(False)

    @torch.no_grad()
    def update_target(self):
        tau = self.ema_decay
        for po, pt in zip(self.online_enc.parameters(), self.target_enc.parameters()):
            pt.data.mul_(tau).add_((1-tau)*po.data)
        for po, pt in zip(self.online_proj.parameters(), self.target_proj.parameters()):
            pt.data.mul_(tau).add_((1-tau)*po.data)

    def forward(self, x1, x2):
        h1 = self.online_enc(x1);  z1 = self.online_proj(h1); p1 = self.predictor(z1)
        h2 = self.online_enc(x2);  z2 = self.online_proj(h2); p2 = self.predictor(z2)
        with torch.no_grad():
            tz1 = self.target_proj(self.target_enc(x1))
            tz2 = self.target_proj(self.target_enc(x2))
        return h1, p1, p2, tz1, tz2


def byol_loss(p, z):
    return (2 - 2*F.cosine_similarity(p, z.detach(), dim=-1)).mean()


class LinearClassifier(nn.Module):
    def __init__(self, encoder, emb_dim=256, num_classes=3, freeze=True):
        super().__init__()
        self.encoder = encoder
        if freeze:
            for p in self.encoder.parameters(): p.requires_grad_(False)
        self.cls = nn.Linear(emb_dim, num_classes)
    def forward(self, x): return self.cls(self.encoder(x))


class SupervisedModel(nn.Module):
    def __init__(self, in_ch=8, emb_dim=256, num_classes=3):
        super().__init__()
        self.encoder = Encoder1D(in_ch, emb_dim)
        self.cls     = nn.Linear(emb_dim, num_classes)
    def forward(self, x): return self.cls(self.encoder(x))

print("Models defined.")

In [ ]:
def evaluate(model, loader):
    model.eval(); ys, preds = [], []
    with torch.no_grad():
        for x, yb in loader:
            preds.extend(model(x.to(device)).argmax(1).cpu().tolist())
            ys.extend(yb.tolist())
    acc = accuracy_score(ys, preds)
    f1  = f1_score(ys, preds, average="macro")
    cm  = confusion_matrix(ys, preds)
    rep = classification_report(ys, preds,
            target_names=[ID2LABEL[i] for i in range(NUM_CLASSES)])
    return acc, f1, cm, rep


def plot_cm(cm, title=""):
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=[ID2LABEL[i] for i in range(NUM_CLASSES)],
                yticklabels=[ID2LABEL[i] for i in range(NUM_CLASSES)])
    plt.title(title); plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout(); plt.show()


def make_adamw(params, lr):
    return torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)


def warmup_cosine(opt, total_ep):
    wu = min(WARMUP_EPOCHS, max(1, total_ep-1))
    def lr_fn(ep):
        if ep < wu: return (ep+1)/wu
        p = (ep-wu+1)/max(1, total_ep-wu)
        return MIN_LR_FACTOR + (1-MIN_LR_FACTOR)*0.5*(1+math.cos(math.pi*p))
    return torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)


class LabeledDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

## Temporal Neighbourhood Dataset

Positive pair: **(window_i, window_j)** where *j* is within ±`k` positions
of *i* in the same subject's time sequence. Nearby windows share the same
stress condition (conditions last several minutes). Both views are then
**composed** with 2 augmentations drawn independently.

In [ ]:
class TemporalNeighborDS(Dataset):
    def __init__(self, X, groups, k=TEMPORAL_K):
        self.X  = X.astype(np.float32)
        self.k  = k
        self.neighbors = [None]*len(X)
        for sub in np.unique(groups):
            idx = np.where(groups == sub)[0]
            for pos, gi in enumerate(idx):
                lo = max(0, pos-k); hi = min(len(idx)-1, pos+k)
                cands = idx[lo:hi+1]
                self.neighbors[gi] = cands[cands != gi]

    def __len__(self): return len(self.X)

    def __getitem__(self, i):
        xi  = self.X[i]
        nbr = self.neighbors[i]
        j   = int(np.random.choice(nbr)) if len(nbr) > 0 else i
        xj  = self.X[j]
        # Independent 2-augmentation composition for each view
        return torch.tensor(compose_augment(xi)), torch.tensor(compose_augment(xj))

## BYOL Pretraining

**EMA cosine schedule** (from original BYOL paper):
$$\tau_t = \tau_{\text{end}} - (\tau_{\text{end}} - \tau_{\text{start}})
   \cdot \frac{\cos(\pi\, t / T) + 1}{2}$$

τ starts at 0.990 (target updates faster → quicker early learning)
and rises to 0.999 (target nearly frozen → stable late representations).
Pretraining uses **all 15 subjects** (no labels).

In [ ]:
ssl_ds  = TemporalNeighborDS(X_all_norm, groups_all, k=TEMPORAL_K)
ssl_dl  = DataLoader(ssl_ds, batch_size=SSL_BATCH, shuffle=True,
                     drop_last=True, num_workers=0)
val_ds  = TemporalNeighborDS(X_val, g_val, k=TEMPORAL_K)
val_dl  = DataLoader(val_ds, batch_size=SSL_BATCH, shuffle=False,
                     drop_last=False, num_workers=0)

byol = BYOL(in_ch=N_CH, emb_dim=EMB_DIM, proj_dim=PROJ_DIM,
             pred_dim=PRED_DIM, ema_decay=EMA_START).to(device)

byol_opt   = make_adamw(byol.parameters(), lr=LR_SSL)
byol_sched = warmup_cosine(byol_opt, SSL_EPOCHS)

total_steps   = SSL_EPOCHS * max(1, len(ssl_dl))
byol_history  = {"train_loss": [], "val_loss": [], "lr": [], "tau": []}
best_val      = float("inf")
best_ckpt     = os.path.join(OUTPUT_DIR, "byol_best.pt")
global_step   = 0

for epoch in range(SSL_EPOCHS):
    byol.train(); tr_loss = 0.0

    for x1, x2 in tqdm(ssl_dl, desc=f"BYOL {epoch+1}/{SSL_EPOCHS}", leave=False):
        x1, x2 = x1.to(device), x2.to(device)

        # EMA cosine schedule (per batch)
        tau = (EMA_END - (EMA_END - EMA_START) *
               (math.cos(math.pi * global_step / total_steps) + 1) / 2)
        byol.ema_decay = tau

        _, p1, p2, tz1, tz2 = byol(x1, x2)
        loss = 0.5 * (byol_loss(p1, tz2) + byol_loss(p2, tz1))
        byol_opt.zero_grad(); loss.backward(); byol_opt.step()
        byol.update_target()
        tr_loss += loss.item(); global_step += 1

    byol.eval(); va_loss = 0.0
    with torch.no_grad():
        for x1, x2 in val_dl:
            x1, x2 = x1.to(device), x2.to(device)
            _, p1, p2, tz1, tz2 = byol(x1, x2)
            va_loss += 0.5*(byol_loss(p1,tz2)+byol_loss(p2,tz1)).item()

    tr_loss /= max(1, len(ssl_dl)); va_loss /= max(1, len(val_dl))
    cur_lr   = byol_sched.get_last_lr()[0]; byol_sched.step()

    byol_history["train_loss"].append(tr_loss)
    byol_history["val_loss"].append(va_loss)
    byol_history["lr"].append(cur_lr)
    byol_history["tau"].append(byol.ema_decay)

    if (epoch+1) % 10 == 0 or epoch == 0:
        print(f"[BYOL] ep {epoch+1:03d} | train={tr_loss:.4f} | "
              f"val={va_loss:.4f} | lr={cur_lr:.2e} | tau={byol.ema_decay:.4f}")
    if va_loss < best_val:
        best_val = va_loss; torch.save(byol.state_dict(), best_ckpt)

print(f"\nBest val BYOL loss: {best_val:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(byol_history["train_loss"], label="train", marker="o", markevery=5)
axes[0].plot(byol_history["val_loss"],   label="val",   marker="s", markevery=5)
axes[0].set_title("BYOL loss (train vs val)"); axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BYOL loss"); axes[0].legend()

axes[1].plot(byol_history["lr"], color="orange", marker="o", markevery=5)
axes[1].set_title("Learning-rate (warmup + cosine)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("LR")

axes[2].plot(byol_history["tau"], color="green", marker="o", markevery=5)
axes[2].set_title("EMA decay τ (cosine schedule)")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("τ")
axes[2].axhline(EMA_START, color="gray", linestyle="--", label=f"start={EMA_START}")
axes[2].axhline(EMA_END,   color="gray", linestyle=":",  label=f"end={EMA_END}")
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"byol_training.png"),dpi=150,bbox_inches="tight")
plt.show()

## Downstream Evaluation: Linear Probe, Fine-tune, Supervised

In [ ]:
byol.load_state_dict(torch.load(best_ckpt, map_location=device))

train_dl = DataLoader(LabeledDS(X_train, y_train), SUP_BATCH, shuffle=True)
test_dl  = DataLoader(LabeledDS(X_test,  y_test),  SUP_BATCH, shuffle=False)
criterion = nn.CrossEntropyLoss()


def run_linear(encoder, epochs, lr, freeze):
    m = LinearClassifier(encoder, EMB_DIM, NUM_CLASSES, freeze).to(device)
    o = make_adamw(filter(lambda p: p.requires_grad, m.parameters()), lr)
    s = warmup_cosine(o, epochs)
    hist = []
    for ep in range(epochs):
        m.train()
        for x, yb in train_dl:
            loss = criterion(m(x.to(device)), yb.to(device))
            o.zero_grad(); loss.backward(); o.step()
        s.step()
        acc, f1, _, _ = evaluate(m, test_dl)
        hist.append(acc)
        print(f"  ep {ep+1:02d} | acc={acc:.4f} | f1={f1:.4f}")
    return m, hist


print("\n=== Linear Eval (frozen encoder) ===")
lin_m, lin_hist = run_linear(copy.deepcopy(byol.online_enc),
                               LINEAR_EPOCHS, LR_LINEAR, freeze=True)
byol_acc, byol_f1, byol_cm, byol_rep = evaluate(lin_m, test_dl)
print(f"\nBYOL+Linear => Acc={byol_acc:.4f}  F1={byol_f1:.4f}")
print(byol_rep); plot_cm(byol_cm, "BYOL + Linear Eval")

print("\n=== Fine-tune (unfrozen, LR/10) ===")
byol.load_state_dict(torch.load(best_ckpt, map_location=device))
ft_m, ft_hist = run_linear(copy.deepcopy(byol.online_enc),
                             SUP_EPOCHS, LR_LINEAR*0.1, freeze=False)
ft_acc, ft_f1, ft_cm, ft_rep = evaluate(ft_m, test_dl)
print(f"\nBYOL+Fine-tune => Acc={ft_acc:.4f}  F1={ft_f1:.4f}")
print(ft_rep); plot_cm(ft_cm, "BYOL Fine-tune")

print("\n=== Supervised Baseline (same architecture) ===")
sup_m    = SupervisedModel(N_CH, EMB_DIM, NUM_CLASSES).to(device)
sup_opt  = make_adamw(sup_m.parameters(), LR_SUP)
sup_sch  = warmup_cosine(sup_opt, SUP_EPOCHS)
sup_hist = []
for ep in range(SUP_EPOCHS):
    sup_m.train()
    for x, yb in train_dl:
        loss = criterion(sup_m(x.to(device)), yb.to(device))
        sup_opt.zero_grad(); loss.backward(); sup_opt.step()
    sup_sch.step()
    acc, f1, _, _ = evaluate(sup_m, test_dl)
    sup_hist.append(acc)
    print(f"  ep {ep+1:02d} | acc={acc:.4f} | f1={f1:.4f}")
sup_acc, sup_f1, sup_cm, sup_rep = evaluate(sup_m, test_dl)
print(f"\nSupervised => Acc={sup_acc:.4f}  F1={sup_f1:.4f}")
print(sup_rep); plot_cm(sup_cm, "Supervised Baseline")

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(lin_hist, label="BYOL + Linear", marker="o", markevery=3)
ax.plot(ft_hist,  label="BYOL + Fine-tune", marker="s", markevery=3)
ax.plot(sup_hist, label="Supervised", marker="^", markevery=3)
ax.set_xlabel("Epoch"); ax.set_ylabel("Test Accuracy")
ax.set_title("Downstream accuracy over epochs"); ax.set_ylim(0,1); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"accuracy_curves.png"),dpi=150,bbox_inches="tight")
plt.show()

results_df = pd.DataFrame([
    {"Method":"BYOL + Linear (frozen)", "Accuracy":byol_acc, "Macro-F1":byol_f1},
    {"Method":"BYOL + Fine-tune",       "Accuracy":ft_acc,   "Macro-F1":ft_f1},
    {"Method":"Supervised (scratch)",   "Accuracy":sup_acc,  "Macro-F1":sup_f1},
])
display(results_df.round(4))
results_df.to_csv(os.path.join(OUTPUT_DIR,"main_results.csv"), index=False)

## Ablation 1: Temporal Neighbourhood Size (k)

How far apart (in window steps) can positive pairs be while still sharing
the same stress state?  We retrain BYOL with *k* ∈ {2, 5, 8, 15, ∞}
(∞ = any window from the same subject, i.e. augmentation-only baseline).

In [ ]:
K_VALUES = [2, 5, 8, 15, 9999]   # 9999 ≈ any window in the same subject
K_LABELS = ["k=2","k=5","k=8","k=15","k=∞ (same subj)"]

k_rows = []
for k_val, k_lbl in zip(K_VALUES, K_LABELS):
    print(f"\n=== Temporal K = {k_lbl} ===")
    set_seed(SEED)

    kds = TemporalNeighborDS(X_all_norm, groups_all, k=k_val)
    kdl = DataLoader(kds, SSL_BATCH, shuffle=True, drop_last=True, num_workers=0)

    km  = BYOL(N_CH, EMB_DIM, PROJ_DIM, PRED_DIM, EMA_START).to(device)
    ko  = make_adamw(km.parameters(), LR_SSL)
    ks  = warmup_cosine(ko, SSL_EPOCHS)
    bv, bc = float("inf"), os.path.join(OUTPUT_DIR, f"byol_k{k_val}.pt")
    tot = SSL_EPOCHS * max(1, len(kdl)); gs = 0

    for ep in range(SSL_EPOCHS):
        km.train(); tl = 0.0
        for x1, x2 in kdl:
            x1, x2 = x1.to(device), x2.to(device)
            tau = EMA_END-(EMA_END-EMA_START)*(math.cos(math.pi*gs/tot)+1)/2
            km.ema_decay = tau
            _, p1,p2,tz1,tz2 = km(x1,x2)
            loss=0.5*(byol_loss(p1,tz2)+byol_loss(p2,tz1))
            ko.zero_grad(); loss.backward(); ko.step(); km.update_target()
            tl += loss.item(); gs += 1
        ks.step()
        if (ep+1)%20==0:
            print(f"  ep {ep+1:3d} | train={tl/max(1,len(kdl)):.4f} | tau={km.ema_decay:.4f}")
        if tl/max(1,len(kdl)) < bv:
            bv=tl/max(1,len(kdl)); torch.save(km.state_dict(), bc)

    km.load_state_dict(torch.load(bc, map_location=device))
    le = LinearClassifier(copy.deepcopy(km.online_enc), EMB_DIM, NUM_CLASSES, True).to(device)
    lo = make_adamw(filter(lambda p:p.requires_grad, le.parameters()), LR_LINEAR)
    ls = warmup_cosine(lo, LINEAR_EPOCHS)
    for ep in range(LINEAR_EPOCHS):
        le.train()
        for x,yb in train_dl:
            loss=criterion(le(x.to(device)),yb.to(device))
            lo.zero_grad();loss.backward();lo.step()
        ls.step()
    acc,f1,_,_ = evaluate(le, test_dl)
    k_rows.append({"k":k_lbl,"Accuracy":acc,"Macro-F1":f1})
    print(f"  => Acc={acc:.4f}  F1={f1:.4f}")

k_df = pd.DataFrame(k_rows)
display(k_df.round(4))

fig, ax = plt.subplots(figsize=(7,4))
ax.bar(k_df["k"], k_df["Accuracy"], alpha=0.85, color="steelblue")
ax.axhline(byol_acc, color="navy",   linestyle="--", label=f"Main BYOL (k={TEMPORAL_K})")
ax.axhline(sup_acc,  color="tomato", linestyle="--", label="Supervised")
ax.set_ylim(0,1); ax.set_xlabel("Temporal neighbourhood k")
ax.set_ylabel("Linear eval accuracy"); ax.set_title("Temporal-K sensitivity")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"temporal_k_ablation.png"),dpi=150,bbox_inches="tight")
plt.show()
k_df.to_csv(os.path.join(OUTPUT_DIR,"temporal_k_ablation.csv"), index=False)

## Ablation 2: Augmentation Strategy

Compare four strategies to understand which augmentation design matters most:

| Strategy | Description |
|----------|-------------|
| None | Temporal neighbours, no augmentation |
| Single | 1 random augmentation per view (3-aug pool) |
| Composed-3 | 2 random augmentations from 3-aug pool (jitter/scale/timemask) |
| Composed-6 | **2 random augmentations from full 6-aug pool** (default) |

In [ ]:
class TemporalNeighborDSCustomAug(Dataset):
    """Same as TemporalNeighborDS but with a configurable augmentation callable."""
    def __init__(self, X, groups, aug_fn, k=TEMPORAL_K):
        self.X, self.aug_fn = X.astype(np.float32), aug_fn
        self.neighbors = [None]*len(X)
        for sub in np.unique(groups):
            idx = np.where(groups==sub)[0]
            for pos,gi in enumerate(idx):
                lo=max(0,pos-k); hi=min(len(idx)-1,pos+k)
                c=idx[lo:hi+1]; self.neighbors[gi]=c[c!=gi]
    def __len__(self): return len(self.X)
    def __getitem__(self,i):
        xi=self.X[i]; nbr=self.neighbors[i]
        j=int(np.random.choice(nbr)) if len(nbr)>0 else i; xj=self.X[j]
        return torch.tensor(self.aug_fn(xi)), torch.tensor(self.aug_fn(xj))


POOL3 = [jitter, scaling, time_mask]

AUG_STRATEGIES = {
    "None":       lambda x: x.copy().astype(np.float32),
    "Single-3":   lambda x: random.choice(POOL3)(x.copy()),
    "Composed-3": lambda x: (lambda f1,f2: f2(f1(x.copy())))(*random.sample(POOL3, 2)),
    "Composed-6": compose_augment,           # full 6-aug pool, 2 drawn
}

aug_rows = []
for strat_name, aug_fn in AUG_STRATEGIES.items():
    print(f"\n=== Aug strategy: {strat_name} ===")
    set_seed(SEED)

    ads = TemporalNeighborDSCustomAug(X_all_norm, groups_all, aug_fn)
    adl = DataLoader(ads, SSL_BATCH, shuffle=True, drop_last=True, num_workers=0)

    am  = BYOL(N_CH, EMB_DIM, PROJ_DIM, PRED_DIM, EMA_START).to(device)
    ao  = make_adamw(am.parameters(), LR_SSL)
    as_ = warmup_cosine(ao, SSL_EPOCHS)
    bv, bc = float("inf"), os.path.join(OUTPUT_DIR, f"byol_aug_{strat_name}.pt")
    tot=SSL_EPOCHS*max(1,len(adl)); gs=0

    for ep in range(SSL_EPOCHS):
        am.train(); tl=0.0
        for x1,x2 in adl:
            x1,x2=x1.to(device),x2.to(device)
            tau=EMA_END-(EMA_END-EMA_START)*(math.cos(math.pi*gs/tot)+1)/2
            am.ema_decay=tau
            _,p1,p2,tz1,tz2=am(x1,x2)
            loss=0.5*(byol_loss(p1,tz2)+byol_loss(p2,tz1))
            ao.zero_grad();loss.backward();ao.step();am.update_target()
            tl+=loss.item();gs+=1
        as_.step()
        if (ep+1)%20==0:
            print(f"  ep {ep+1:3d} | train={tl/max(1,len(adl)):.4f}")
        if tl/max(1,len(adl))<bv:
            bv=tl/max(1,len(adl));torch.save(am.state_dict(),bc)

    am.load_state_dict(torch.load(bc, map_location=device))
    le=LinearClassifier(copy.deepcopy(am.online_enc),EMB_DIM,NUM_CLASSES,True).to(device)
    lo=make_adamw(filter(lambda p:p.requires_grad,le.parameters()),LR_LINEAR)
    ls=warmup_cosine(lo,LINEAR_EPOCHS)
    for ep in range(LINEAR_EPOCHS):
        le.train()
        for x,yb in train_dl:
            loss=criterion(le(x.to(device)),yb.to(device))
            lo.zero_grad();loss.backward();lo.step()
        ls.step()
    acc,f1,_,_=evaluate(le,test_dl)
    aug_rows.append({"Augmentation Strategy":strat_name,"Accuracy":acc,"Macro-F1":f1})
    print(f"  => Acc={acc:.4f}  F1={f1:.4f}")

aug_df=pd.DataFrame(aug_rows)
display(aug_df.round(4))

fig,axes=plt.subplots(1,2,figsize=(11,4))
for ax,col in zip(axes,["Accuracy","Macro-F1"]):
    ax.bar(aug_df["Augmentation Strategy"], aug_df[col], alpha=0.85)
    ax.axhline(sup_acc if col=="Accuracy" else sup_f1,
               color="tomato",linestyle="--",label="Supervised")
    ax.set_ylim(0,1); ax.set_title(f"Aug strategy ablation: {col}")
    ax.set_xticklabels(aug_df["Augmentation Strategy"], rotation=15)
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"aug_strategy_ablation.png"),dpi=150,bbox_inches="tight")
plt.show()
aug_df.to_csv(os.path.join(OUTPUT_DIR,"aug_strategy_ablation.csv"),index=False)

## Low-label Study

SSL's clearest advantage: with few labels, BYOL+Linear beats supervised from scratch.
The gap closes as more labels become available.

In [ ]:
def sample_frac(X, y, frac):
    if frac >= 1.0: return X, y
    k = max(1, int(len(X)*frac))
    idx = np.random.default_rng(SEED).choice(len(X), k, replace=False)
    return X[idx], y[idx]

byol.load_state_dict(torch.load(best_ckpt, map_location=device))
fracs = [0.05, 0.10, 0.25, 0.50, 1.00]
ll_rows = []

for frac in fracs:
    Xs, ys = sample_frac(X_train, y_train, frac)
    sdl = DataLoader(LabeledDS(Xs, ys), SUP_BATCH, shuffle=True)

    # BYOL + Linear
    le=LinearClassifier(copy.deepcopy(byol.online_enc),EMB_DIM,NUM_CLASSES,True).to(device)
    lo=make_adamw(filter(lambda p:p.requires_grad,le.parameters()),LR_LINEAR)
    ls=warmup_cosine(lo,LINEAR_EPOCHS)
    for ep in range(LINEAR_EPOCHS):
        le.train()
        for x,yb in sdl:
            loss=criterion(le(x.to(device)),yb.to(device))
            lo.zero_grad();loss.backward();lo.step()
        ls.step()
    acc,f1,_,_=evaluate(le,test_dl)
    ll_rows.append({"Method":"BYOL+Linear","Label Fraction":frac,"Accuracy":acc,"Macro-F1":f1})

    # Supervised
    sm=SupervisedModel(N_CH,EMB_DIM,NUM_CLASSES).to(device)
    so=make_adamw(sm.parameters(),LR_SUP); ss=warmup_cosine(so,SUP_EPOCHS)
    for ep in range(SUP_EPOCHS):
        sm.train()
        for x,yb in sdl:
            loss=criterion(sm(x.to(device)),yb.to(device))
            so.zero_grad();loss.backward();so.step()
        ss.step()
    acc,f1,_,_=evaluate(sm,test_dl)
    ll_rows.append({"Method":"Supervised","Label Fraction":frac,"Accuracy":acc,"Macro-F1":f1})
    print(f"  frac={frac:.0%} done")

ll_df=pd.DataFrame(ll_rows)
display(ll_df.round(4))

plt.figure(figsize=(7,4))
for method in ll_df["Method"].unique():
    d=ll_df[ll_df["Method"]==method].sort_values("Label Fraction")
    plt.plot(d["Label Fraction"],d["Accuracy"],marker="o",label=method)
plt.xlabel("Label fraction"); plt.ylabel("Test accuracy")
plt.title("Label efficiency: BYOL vs Supervised"); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"low_label.png"),dpi=150,bbox_inches="tight")
plt.show()
ll_df.to_csv(os.path.join(OUTPUT_DIR,"low_label.csv"),index=False)

## Comprehensive Results Summary

In [ ]:
print("=" * 60)
print("MAIN RESULTS")
print("=" * 60)
display(results_df.round(4))

print("\nTEMPORAL-K ABLATION")
display(k_df.round(4))

print("\nAUGMENTATION STRATEGY ABLATION")
display(aug_df.round(4))

print("\nLOW-LABEL STUDY")
display(ll_df.pivot(index="Label Fraction",columns="Method",values="Accuracy").round(4))

## Save Outputs

In [ ]:
torch.save(byol.state_dict(),  os.path.join(OUTPUT_DIR,"byol_pretrained.pt"))
torch.save(ft_m.state_dict(),  os.path.join(OUTPUT_DIR,"byol_finetuned.pt"))
torch.save(sup_m.state_dict(), os.path.join(OUTPUT_DIR,"supervised.pt"))
print("Models saved.")
print("CSVs:", [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".csv")])

## Discussion

### Why these choices improve over SimCLR

**Composed augmentation (2 of 6)** forces the encoder to learn invariance to multiple
simultaneous distortions.  Using only one augmentation at a time is too easy —
the encoder can identify the view type rather than the stress state.
`freq_mask` is particularly important for ECG because it removes specific harmonic
components, forcing the encoder to rely on the full spectral structure.
`gaussian_smooth` simulates lower-quality wrist sensors.

**Temporal neighbourhood positives** encode a physiologically-grounded prior:
stress and amusement each last several minutes, so windows within ±8 positions
(±2 min) share the same state.  The Temporal-K ablation finds the optimal k.

**EMA cosine schedule** (τ: 0.99→0.999) lets the target network update more
aggressively early in training (when representations are noisy) and stabilise
late (when representations are converging).  A fixed τ misses one of these phases.

**All-subject pretraining** is the single largest data multiplier: the encoder
must learn features that generalise across 15 different subjects' physiology,
which prevents subject-identity memorisation.

### Where BYOL should beat supervised
- **Low-label regime (5–25 %)**: BYOL+Linear uses a rich pretrained representation
  while supervised starts from scratch with few examples.
- **Fine-tuning (100 % labels)**: BYOL+Fine-tune should match or exceed supervised
  because the SSL initialisation already captures physiological structure.